In [2]:
import numpy as np

In [3]:
X = np.random.randn(3, 4)

W = np.random.randn(4, 2)

b = np.zeros((1, 2))

print(X)

print(W)

print(b)

[[ 0.2057607  -2.10122133  0.36338647  0.85615624]
 [ 0.50582978  0.21160203 -0.56735345  1.30386198]
 [ 0.61263982 -0.21550026  0.57276221  0.14547232]]
[[ 1.44842052 -0.37550586]
 [-0.81474433  3.17159741]
 [ 0.24294464  0.80509964]
 [-0.94316863 -2.49020776]]
[[0. 0.]]


In [4]:
Z = np.dot(X, W) + b

print(Z.shape)
print(Z)

(3, 2)
[[ 1.29076926 -8.58093708]
 [-0.80734453 -3.22248886]
 [ 1.06488228 -0.81465557]]


In [5]:
def ReLU(x):
    return np.maximum(x, 0)

A = ReLU(Z)

print(A)

[[1.29076926 0.        ]
 [0.         0.        ]
 [1.06488228 0.        ]]


In [6]:
dA = np.random.randn(A.shape[0], A.shape[1])

dZ = dA * (Z > 0)

print(dA)

print(dZ)

print(dZ.shape)

[[ 0.60985782 -0.08737104]
 [ 0.46218177  0.94279574]
 [ 1.21371864 -0.23805136]]
[[ 0.60985782 -0.        ]
 [ 0.          0.        ]
 [ 1.21371864 -0.        ]]
(3, 2)


In [7]:
dW = np.dot(X.T, dZ)
print(dW.shape)
print(dW.shape == W.shape)

(4, 2)
True


In [8]:
db = np.sum(dZ, axis=0, keepdims=True)
print(db.shape)
print(db.shape == b.shape)

(1, 2)
True


In [9]:
dX = np.dot(dZ, W.T)
print(dX.shape)
print(dX.shape == X.shape)

(3, 4)
True


In [10]:
class DenseLayer:
    def __init__(self, input_dim, output_dim):
        self.W = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        self.b = np.zeros((1, output_dim))

    def forward(self, X):
        Z = np.dot(X, self.W) + self.b
        self.X = X
        return Z
    
    def backward(self, dZ):
        dW = np.dot(self.X.T, dZ)
        db = np.sum(dZ, axis=0, keepdims=True)
        dX = np.dot(dZ, self.W.T)
        self.dW = dW
        self.db = db
        return dX

class ReLULayer:
    def forward(self, Z):
        self.Z = Z
        return np.maximum(Z, 0)
    
    def backward(self, dA):
        return dA * (self.Z > 0)

In [11]:
class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, loss_gradient):
        for layer in self.layers[::-1]:
            loss_gradient = layer.backward(loss_gradient)

In [12]:
# testing out nn

nn = NeuralNetwork([
    DenseLayer(input_dim=4, output_dim=10),
    ReLULayer(),
    DenseLayer(input_dim=10, output_dim=2)
])

X_test = np.random.randn(3, 4)
predictions = nn.forward(X_test)
print("Output Shape (Should be 3, 2):", predictions.shape)

fake_loss_gradient = np.random.randn(3, 2)
nn.backward(fake_loss_gradient)

print("First layer dW shape (Should be 4, 10):", nn.layers[0].dW.shape)
print("First layer db shape (Should be 1, 10):", nn.layers[0].db.shape)

Output Shape (Should be 3, 2): (3, 2)
First layer dW shape (Should be 4, 10): (4, 10)
First layer db shape (Should be 1, 10): (1, 10)


In [13]:
# create update parameters function

class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, loss_gradient):
        for layer in self.layers[::-1]:
            loss_gradient = layer.backward(loss_gradient)

    def update_params(self, lr):
        for layer in self.layers:
            if hasattr(layer, 'W'):
                layer.W -= lr * layer.dW
                layer.b -= lr * layer.db

In [14]:
# testing out full neural network

nn = NeuralNetwork([
    DenseLayer(input_dim=4, output_dim=10),
    ReLULayer(),
    DenseLayer(input_dim=10, output_dim=2)
])

X_test = np.random.randn(3, 4)
Y_true = [[1, 0], [0, 1], [1, 0]]

epoch = 10
lr = 0.01

for i in range(1, epoch+1):
    Y_pred = nn.forward(X_test)
    loss_gradient = Y_pred - Y_true
    loss_value = np.mean((Y_pred - Y_true) ** 2) / 2
    nn.backward(loss_gradient)
    nn.update_params(lr)
    if not i % 10 or i == 1:
        print(f'Epoch {i} | Loss: {loss_value:.6f}')

Epoch 1 | Loss: 1.104413
Epoch 10 | Loss: 0.121985


In [15]:
# load handwritten digits classifer data

with np.load('mnist.npz') as data:
    x_train = data['x_train']
    y_train = data['y_train']
    x_test = data['x_test']
    y_test = data['y_test']

# matrix reshaping and normalization
x_train= x_train.reshape(60000, -1) / 255.0
x_test= x_test.reshape(10000, -1) / 255.0

# one hot encoding
y_train = np.eye(10)[y_train]
y_test = np.eye(10)[y_test]

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(60000, 784)
(10000, 784)
(60000, 10)
(10000, 10)


In [16]:
# softmax makes output fall between 0 and 1

class SoftmaxLayer:
    def forward(self, Z):
        shift_Z = Z - np.max(Z, axis=-1, keepdims=True)
        exps = np.exp(shift_Z)
        return exps / np.sum(exps, axis=-1, keepdims=True)
    
    def backward(self, dA):
        return dA

In [17]:
# calculate loss gradients

class CrossEntropyLoss:
    def forward(self, Y_pred, Y_true):
        self.Y_pred = Y_pred
        self.Y_true = Y_true
        loss = -np.sum(Y_true * np.log(Y_pred + 1e-15)) / Y_pred.shape[0]
        return loss
    
    def backward(self):
        return (self.Y_pred - self.Y_true) / self.Y_pred.shape[0]

In [18]:
def accuracy(Y_pred, Y_true):
    predicted_digits = np.argmax(Y_pred, axis=1)
    true_digits = np.argmax(Y_true, axis=1)
    return np.mean(predicted_digits == true_digits) * 100

In [19]:
# simple classifier with 1 hidden layer of 128 neurons, 10 epochs and 0.05 lr

nn = NeuralNetwork([
    DenseLayer(input_dim=784, output_dim=128),
    ReLULayer(),
    DenseLayer(input_dim=128, output_dim=10),
    SoftmaxLayer()
])

loss_fn = CrossEntropyLoss()

epochs = 10
batch_size = 128
lr = 0.05

for epoch in range(1, epochs+1):
    indices = np.arange(x_train.shape[0])
    np.random.shuffle(indices)

    total_loss = 0
    num_batches = x_train.shape[0] / batch_size

    for i in range(0, x_train.shape[0], batch_size):
        batch_indices = indices[i:i+batch_size]
        x_batch = x_train[batch_indices]
        y_batch = y_train[batch_indices]
        
        y_pred = nn.forward(x_batch)
        
        loss = loss_fn.forward(y_pred, y_batch)
        total_loss += loss
        
        loss_gradient = loss_fn.backward()
        nn.backward(loss_gradient)
        nn.update_params(lr)
        
    average_epoch_loss = total_loss / (x_train.shape[0] / batch_size)
    print(f"Epoch {epoch} completed | Average Loss: {average_epoch_loss:.4f}")

test_preds = nn.forward(x_test)
final_acc = accuracy(test_preds, y_test)
print(f"Final Test Accuracy: {final_acc:.2f}%")

Epoch 1 completed | Average Loss: 0.5864
Epoch 2 completed | Average Loss: 0.3163
Epoch 3 completed | Average Loss: 0.2696
Epoch 4 completed | Average Loss: 0.2387
Epoch 5 completed | Average Loss: 0.2154
Epoch 6 completed | Average Loss: 0.1967
Epoch 7 completed | Average Loss: 0.1811
Epoch 8 completed | Average Loss: 0.1681
Epoch 9 completed | Average Loss: 0.1567
Epoch 10 completed | Average Loss: 0.1469
Final Test Accuracy: 95.77%


In [21]:
# complex classifier with many hidden layer with more neurons, 100 epochs and 0.01 lr

nn = NeuralNetwork([
    DenseLayer(input_dim=784, output_dim=512),
    ReLULayer(),
    DenseLayer(input_dim=512, output_dim=256),
    ReLULayer(),
    DenseLayer(input_dim=256, output_dim=128),
    ReLULayer(),
    DenseLayer(input_dim=128, output_dim=64),
    ReLULayer(),
    DenseLayer(input_dim=64, output_dim=10),
    SoftmaxLayer()
])

loss_fn = CrossEntropyLoss()

epochs = 100
batch_size = 128
lr = 0.01

for epoch in range(1, epochs+1):
    indices = np.arange(x_train.shape[0])
    np.random.shuffle(indices)

    total_loss = 0
    num_batches = x_train.shape[0] / batch_size

    for i in range(0, x_train.shape[0], batch_size):
        batch_indices = indices[i:i+batch_size]
        x_batch = x_train[batch_indices]
        y_batch = y_train[batch_indices]
        
        y_pred = nn.forward(x_batch)
        
        loss = loss_fn.forward(y_pred, y_batch)
        total_loss += loss
        
        loss_gradient = loss_fn.backward()
        nn.backward(loss_gradient)
        nn.update_params(lr)
        
    average_epoch_loss = total_loss / (x_train.shape[0] / batch_size)
    if not epoch%10 or epoch==1:
        print(f"Epoch {epoch} completed | Average Loss: {average_epoch_loss:.4f}")

test_preds = nn.forward(x_test)
final_acc = accuracy(test_preds, y_test)
print(f"Final Test Accuracy: {final_acc:.2f}%")

Epoch 1 completed | Average Loss: 0.8558
Epoch 10 completed | Average Loss: 0.1274
Epoch 20 completed | Average Loss: 0.0688
Epoch 30 completed | Average Loss: 0.0397
Epoch 40 completed | Average Loss: 0.0232
Epoch 50 completed | Average Loss: 0.0137
Epoch 60 completed | Average Loss: 0.0083
Epoch 70 completed | Average Loss: 0.0054
Epoch 80 completed | Average Loss: 0.0038
Epoch 90 completed | Average Loss: 0.0028
Epoch 100 completed | Average Loss: 0.0021
Final Test Accuracy: 97.79%


In [22]:
import pickle

# saving nn
with open('mnist_nn_model.pkl', 'wb') as f:
    pickle.dump(nn, f)

print("Model saved successfully as 'mnist_nn_model.pkl'!")

Model saved successfully as 'mnist_nn_model.pkl'!
